# M3 decision reports on Kaggle

Runs `serpentguard.decision.decision_report` on every training run whose kernel output is attached to this notebook, on
the CPU. For each run it fits the verdict thresholds on the calibration outputs, reports the verdicts on the
validation and unknown-pool outputs, and writes `decision_metrics.json`. A summary table of the main verdict numbers
goes to `decision_summary.csv`.

`SCORER` picks what ranks the species, the species head (`"head"`) or the nearest prototype (`"prototype"`,
plan section 2.1), for the fit and every verdict; the notebook can be run once with each. With `PREDS_KERNEL`, a
run that reads predicted masks or boxes (`mask_source: predicted`, F1/F1m/F2/F3) gets `tau_snake` from the front
end's box confidences, so its report counts the photos with no snake found. `RECALL_BOUND` (R49) is `"point"`
to fit `theta_d` and `theta_who` to 95% WHO recall on the calibration set, or `"lower"` to fit them so the
photographer-bootstrap lower bound of that recall reaches 95%; validation never enters the fit.

It runs on Kaggle rather than locally so the saved outputs (`*.npz`) never have to be downloaded. A kernel can read
only its own account's private outputs, so each account runs its own copy; every kernel handle also takes
`"dataset:owner/slug"`, a private dataset copy of another account's output. No GPU is used.

In [ ]:
RUN_KERNELS = ['jayeshbansal497/sg-m3-r3', 'jayeshbansal497/sg-m3-r4']
QUALITY_KERNEL = None   # optional "owner/slug" whose output holds quality.csv (Task 10), for the poor_image reason, or "dataset:owner/slug"
PREDS_KERNEL = 'dataset:jayeshbansal497/serpentguard-m3-frontend-boxes'
SCORER = 'head'
RECALL_BOUND = 'lower'

In [ ]:
import subprocess
import sys

SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt, a Global Constraint: every metric uses it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, f"scikit-learn {sklearn.__version__} is loaded"

In [ ]:
import json
import shutil
from pathlib import Path

import kagglehub
import pandas as pd

OWNER = "puneetbajajiitm"
CODE = Path("/kaggle/input/serpentguard-code/src")
sys.path.insert(0, str(CODE if CODE.exists() else Path("../src").resolve()))

from serpentguard.config import load_config
from serpentguard.decision import decision_report
from serpentguard.run_data import FRONTEND_PREDS_FILE, PREDICTED

RUN_FILES = ("val_outputs.npz", "calib_outputs.npz", "unknown_outputs.npz", "prototypes.npz", "final_metrics.json",
             "config.yaml")
OUT = Path("/kaggle/working/decision")

data_root = Path(kagglehub.dataset_download(f"{OWNER}/serpentguard-indian-snakes-clean"))
classes = pd.read_csv(data_root / "classes.csv")
metadata = pd.read_csv(data_root / "metadata.csv", usecols=["path", "split"])


DATASET_PREFIX = "dataset:"  # "dataset:owner/slug": a private dataset copy of a kernel's output, for another account


def kernel_output(handle):
    """Where a kernel's output is: "owner/slug" reads it (a batch kernel only once it is declared a kernel source),
    "dataset:owner/slug" a private dataset copy of it."""
    if handle.startswith(DATASET_PREFIX):
        return Path(kagglehub.dataset_download(handle.removeprefix(DATASET_PREFIX)))
    return Path(kagglehub.notebook_output_download(handle))


def run_folders(root):
    return sorted(p.parent for p in root.rglob("val_outputs.npz") if (p.parent / "calib_outputs.npz").exists())


quality = None
if QUALITY_KERNEL:
    found = sorted(kernel_output(QUALITY_KERNEL).rglob("quality.csv"))
    quality = pd.read_csv(found[0]) if found else None
    print("quality table:", found[0] if found else "not found")

box_conf = None
if PREDS_KERNEL:
    preds_files = sorted(kernel_output(PREDS_KERNEL).rglob(FRONTEND_PREDS_FILE))
    assert len(preds_files) == 1, f"{PREDS_KERNEL}'s output must hold one {FRONTEND_PREDS_FILE}; found {preds_files}"
    preds = pd.read_csv(preds_files[0], usecols=["path", "box_conf"])
    box_conf = dict(zip(preds["path"], preds["box_conf"]))
    print(f"front-end predictions: {preds_files[0]}, {preds['box_conf'].notna().sum():,} photos with a box")

In [ ]:
rows = []
for handle in RUN_KERNELS:
    for source in run_folders(kernel_output(handle)):
        run = OUT / source.name
        run.mkdir(parents=True, exist_ok=True)
        for name in RUN_FILES:
            if (source / name).exists():
                shutil.copy2(source / name, run / name)
        predicted = load_config(run / "config.yaml").mask_source == PREDICTED
        if predicted and box_conf is None:
            print(f"{source.name} reads predicted masks or boxes; without PREDS_KERNEL it gets no tau_snake")
        report = decision_report(run, classes, metadata=metadata, quality=quality, scorer=SCORER,
                                 box_conf=box_conf if predicted else None, recall_bound=RECALL_BOUND)
        for name in RUN_FILES:
            if name.endswith(".npz"):
                (run / name).unlink()
        metrics, fit = report["verdict_metrics"], report["fit"]
        rows.append({"run": source.name, "kernel": handle, "scorer": report["scorer"],
                     "recall_bound": report["recall_bound"],
                     "recall_target": report.get("recall_bound_fit", {}).get("target"),
                     "who_dangerous_recall": metrics["who_dangerous_recall"], "false_alarm": metrics["false_alarm"],
                     "venomous_recall": metrics["venomous_recall"],
                     "o3_all": metrics["o3"]["dangerous_as_not_listed_all"], "abstain_rate": metrics["abstain_rate"],
                     "unknown_flagged": report["u1"]["groups"]["all"]["flagged_at_tau"],
                     "unknown_auroc_msp": report["u1"]["groups"]["all"]["open_set"]["msp"]["auroc"],
                     "theta_d": report["thresholds"]["theta_d"], "theta_who": report["thresholds"]["theta_who"],
                     "calibration_recall": fit["who_dangerous_recall"],
                     "tau_snake": report["no_snake"].get("tau_snake"), "no_snake_val": report["no_snake"]["val"]})
        print(f"{source.name}: WHO recall {metrics['who_dangerous_recall']:.3f}, false alarms "
              f"{metrics['false_alarm']:.3f}, unknown flagged {rows[-1]['unknown_flagged']:.3f}")
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "decision_summary.csv", index=False)
assert len(summary), "no run folders found in the attached kernel outputs"
summary